# 🎯 CLIP ReID Video Processor

**Standalone video processing with text-based person tracking**

This notebook allows you to:
- Load a video clip and text description
- Process the entire video with CLIP-based person selection
- Track multiple entities with consistent IDs
- Generate output video with tracking visualization

**No Flask server required!** 🚀

## 📦 Setup and Imports

In [ ]:
# Install required packages (run once)
!pip install torch torchvision opencv-python pillow numpy matplotlib scikit-learn
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import cv2
import numpy as np
import torch
import clip
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import Video, display, HTML
import ipywidgets as widgets
from pathlib import Path
import time
import json
from collections import defaultdict, deque
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings('ignore')

# Import our standalone processor
from standalone_clip_reid_notebook import StandaloneCLIPReID

print("✅ All imports successful!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🖥️  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

## 🚀 Initialize CLIP ReID Processor

In [ ]:
# Initialize the processor
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = StandaloneCLIPReID(device=device)

print(f"🎯 CLIP ReID Processor initialized on {device}")
print(f"📊 CLIP threshold: {processor.clip_threshold}")
print(f"🔄 ReID threshold: {processor.reid_threshold}")

## 📹 Video Input Configuration

In [ ]:
# Configure your video and text query
VIDEO_PATH = "path/to/your/video.mp4"  # 👈 UPDATE THIS PATH
TEXT_QUERY = "person wearing red shirt and blue jeans"  # 👈 UPDATE THIS DESCRIPTION
OUTPUT_PATH = "tracked_output.mp4"

# Verify video exists
if Path(VIDEO_PATH).exists():
    print(f"✅ Video found: {VIDEO_PATH}")
    
    # Get video info
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    cap.release()
    
    print(f"📊 Video Info:")
    print(f"   Resolution: {width}x{height}")
    print(f"   FPS: {fps}")
    print(f"   Duration: {duration:.2f} seconds")
    print(f"   Total frames: {total_frames}")
    print(f"🎯 Text Query: '{TEXT_QUERY}'")
else:
    print(f"❌ Video not found: {VIDEO_PATH}")
    print("Please update VIDEO_PATH with a valid video file.")

## 🎛️ Processing Configuration

In [ ]:
# Adjust processing parameters
print("🔧 Adjusting processing parameters...")

# CLIP matching threshold (lower = more matches, higher = stricter)
processor.clip_threshold = 0.25

# ReID matching threshold (higher = better tracking consistency)
processor.reid_threshold = 0.7

# Maximum frames a track can disappear before being removed
processor.max_disappeared = 10

# Number of features to remember per track
processor.feature_memory_size = 5

print(f"✅ Configuration updated:")
print(f"   CLIP threshold: {processor.clip_threshold}")
print(f"   ReID threshold: {processor.reid_threshold}")
print(f"   Max disappeared: {processor.max_disappeared}")
print(f"   Feature memory: {processor.feature_memory_size}")

## 🎬 Process Video

In [ ]:
# Process the video
if Path(VIDEO_PATH).exists():
    print("🚀 Starting video processing...")
    print(f"📹 Input: {VIDEO_PATH}")
    print(f"💾 Output: {OUTPUT_PATH}")
    print(f"🎯 Query: '{TEXT_QUERY}'")
    print("\n" + "="*50)
    
    # Process the video
    stats = processor.process_video(
        video_path=VIDEO_PATH,
        text_query=TEXT_QUERY,
        output_path=OUTPUT_PATH,
        display=False  # Set to True if you want to see real-time processing
    )
    
    print("\n" + "="*50)
    print("🎉 Processing completed!")
    
else:
    print("❌ Cannot process: Video file not found")
    stats = None

## 📊 Results and Statistics

In [ ]:
if stats:
    print("📊 PROCESSING STATISTICS")
    print("="*40)
    
    # Display statistics
    print(f"🎬 Total frames processed: {stats['processed_frames']:,}")
    print(f"⏱️  Processing time: {stats['processing_time']:.2f} seconds")
    print(f"🚀 Average FPS: {stats['fps']:.2f}")
    print(f"👥 Total detections: {stats['total_detections']:,}")
    print(f"🎯 CLIP matches: {stats['clip_matches']:,}")
    print(f"🆔 Unique tracks: {stats['unique_tracks']}")
    
    # Calculate match rate
    if stats['total_detections'] > 0:
        match_rate = (stats['clip_matches'] / stats['total_detections']) * 100
        print(f"📈 CLIP match rate: {match_rate:.1f}%")
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Bar chart of statistics
    categories = ['Total\nDetections', 'CLIP\nMatches', 'Unique\nTracks']
    values = [stats['total_detections'], stats['clip_matches'], stats['unique_tracks']]
    colors = ['skyblue', 'lightgreen', 'orange']
    
    axes[0].bar(categories, values, color=colors)
    axes[0].set_title('Detection Statistics')
    axes[0].set_ylabel('Count')
    
    # Add value labels on bars
    for i, v in enumerate(values):
        axes[0].text(i, v + max(values)*0.01, str(v), ha='center', va='bottom')
    
    # Processing performance
    perf_categories = ['Processing\nTime (s)', 'Average\nFPS']
    perf_values = [stats['processing_time'], stats['fps']]
    perf_colors = ['lightcoral', 'lightblue']
    
    axes[1].bar(perf_categories, perf_values, color=perf_colors)
    axes[1].set_title('Performance Metrics')
    axes[1].set_ylabel('Value')
    
    # Add value labels
    for i, v in enumerate(perf_values):
        axes[1].text(i, v + max(perf_values)*0.01, f'{v:.1f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("❌ No statistics available - video processing failed")

## 🎥 View Output Video

In [ ]:
# Display the output video
if Path(OUTPUT_PATH).exists():
    print(f"🎬 Displaying output video: {OUTPUT_PATH}")
    
    # Show video player
    display(Video(OUTPUT_PATH, width=800, height=600))
    
    # File info
    file_size = Path(OUTPUT_PATH).stat().st_size / (1024*1024)  # MB
    print(f"📁 File size: {file_size:.2f} MB")
    
else:
    print(f"❌ Output video not found: {OUTPUT_PATH}")

## 🔍 Frame-by-Frame Analysis (Optional)

In [ ]:
# Analyze specific frames
def analyze_frame(video_path, frame_number, text_query):
    """Analyze a specific frame"""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print(f"❌ Could not read frame {frame_number}")
        return
    
    # Process frame
    detections = processor.detect_persons(frame)
    
    if detections:
        crops = processor.extract_person_crops(frame, detections)
        text_features = processor.encode_text_query(text_query)
        image_features = processor.encode_person_crops(crops)
        clip_matches = processor.find_clip_matches(text_features, image_features, detections)
        
        # Visualize
        vis_frame = frame.copy()
        
        for i, detection in enumerate(detections):
            bbox = detection['bbox']
            x1, y1, x2, y2 = bbox
            
            # Check if this is a CLIP match
            is_match = any(match[0] == i for match in clip_matches)
            color = (0, 255, 0) if is_match else (255, 0, 0)
            
            cv2.rectangle(vis_frame, (x1, y1), (x2, y2), color, 2)
            
            if is_match:
                score = next(match[1] for match in clip_matches if match[0] == i)
                label = f"CLIP: {score:.3f}"
                cv2.putText(vis_frame, label, (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        # Display
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(vis_frame, cv2.COLOR_BGR2RGB))
        plt.title(f"Frame {frame_number} - Query: '{text_query}'")
        plt.axis('off')
        plt.show()
        
        print(f"🔍 Frame {frame_number} analysis:")
        print(f"   👥 Detections: {len(detections)}")
        print(f"   🎯 CLIP matches: {len(clip_matches)}")
        
        for i, (det_idx, score, detection) in enumerate(clip_matches):
            print(f"   Match {i+1}: Score {score:.3f}")
    
    else:
        print(f"👥 No persons detected in frame {frame_number}")

# Example: Analyze frame 100
if Path(VIDEO_PATH).exists():
    analyze_frame(VIDEO_PATH, 100, TEXT_QUERY)

## 🎯 Multiple Query Processing

In [ ]:
# Process video with multiple text queries
queries = [
    "person wearing red shirt",
    "woman in blue dress",
    "man with black jacket",
    "person wearing white t-shirt"
]

if Path(VIDEO_PATH).exists():
    print("🎯 Processing multiple queries...")
    
    results = []
    
    for i, query in enumerate(queries):
        print(f"\n🔄 Processing query {i+1}/{len(queries)}: '{query}'")
        
        output_file = f"output_query_{i+1}.mp4"
        
        # Reset processor for new query
        processor.person_tracks = {}
        processor.next_track_id = 1
        processor.feature_memory = defaultdict(list)
        processor.track_history = defaultdict(lambda: deque(maxlen=30))
        
        stats = processor.process_video(
            video_path=VIDEO_PATH,
            text_query=query,
            output_path=output_file,
            display=False
        )
        
        results.append({
            'query': query,
            'output_file': output_file,
            'clip_matches': stats['clip_matches'],
            'unique_tracks': stats['unique_tracks']
        })
        
        print(f"✅ Query completed: {stats['clip_matches']} matches, {stats['unique_tracks']} tracks")
    
    # Summary
    print("\n📊 MULTI-QUERY SUMMARY")
    print("="*50)
    
    for result in results:
        print(f"Query: '{result['query']}'")
        print(f"  🎯 Matches: {result['clip_matches']}")
        print(f"  🆔 Tracks: {result['unique_tracks']}")
        print(f"  📁 Output: {result['output_file']}")
        print()
    
    # Visualization
    fig, ax = plt.subplots(figsize=(12, 6))
    
    query_names = [f"Q{i+1}" for i in range(len(queries))]
    matches = [r['clip_matches'] for r in results]
    tracks = [r['unique_tracks'] for r in results]
    
    x = np.arange(len(query_names))
    width = 0.35
    
    ax.bar(x - width/2, matches, width, label='CLIP Matches', color='lightgreen')
    ax.bar(x + width/2, tracks, width, label='Unique Tracks', color='lightblue')
    
    ax.set_xlabel('Queries')
    ax.set_ylabel('Count')
    ax.set_title('Multi-Query Results Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(query_names)
    ax.legend()
    
    # Add value labels
    for i, (m, t) in enumerate(zip(matches, tracks)):
        ax.text(i - width/2, m + max(matches)*0.01, str(m), ha='center', va='bottom')
        ax.text(i + width/2, t + max(tracks)*0.01, str(t), ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Create legend for queries
    print("📝 Query Legend:")
    for i, query in enumerate(queries):
        print(f"  Q{i+1}: '{query}'")

else:
    print("❌ Cannot process multiple queries: Video file not found")

## 💡 Tips and Best Practices

### 🎯 Writing Effective Text Queries

**Good Examples:**
- `"person wearing red shirt"`
- `"woman in blue dress"`
- `"man with black jacket and jeans"`
- `"person wearing glasses and white t-shirt"`

**Tips:**
- Be specific but not overly detailed
- Focus on visible clothing and accessories
- Mention distinctive features (glasses, hat, etc.)
- Avoid subjective descriptions ("beautiful", "tall")

### ⚙️ Parameter Tuning

**CLIP Threshold (0.15 - 0.35):**
- Lower = More matches, less precise
- Higher = Fewer matches, more precise

**ReID Threshold (0.6 - 0.9):**
- Lower = More track connections, potential errors
- Higher = Fewer connections, better accuracy

### 🎬 Video Quality Tips

- **Resolution:** Higher resolution works better
- **Lighting:** Good lighting improves results
- **Occlusion:** Avoid heavily occluded scenes
- **Motion:** Moderate motion works best

### 🚀 Performance Optimization

- Use GPU when available
- Process shorter clips for testing
- Adjust frame skip for faster processing
- Lower video resolution if needed

## 🔧 Advanced Configuration

In [ ]:
# Advanced configuration options
print("🔧 Advanced Configuration Options")
print("="*40)

# Create interactive widgets for parameter tuning
clip_threshold_slider = widgets.FloatSlider(
    value=processor.clip_threshold,
    min=0.1,
    max=0.5,
    step=0.05,
    description='CLIP Threshold:',
    style={'description_width': 'initial'}
)

reid_threshold_slider = widgets.FloatSlider(
    value=processor.reid_threshold,
    min=0.5,
    max=0.95,
    step=0.05,
    description='ReID Threshold:',
    style={'description_width': 'initial'}
)

max_disappeared_slider = widgets.IntSlider(
    value=processor.max_disappeared,
    min=5,
    max=30,
    step=1,
    description='Max Disappeared:',
    style={'description_width': 'initial'}
)

def update_parameters(clip_thresh, reid_thresh, max_disap):
    processor.clip_threshold = clip_thresh
    processor.reid_threshold = reid_thresh
    processor.max_disappeared = max_disap
    print(f"Updated: CLIP={clip_thresh}, ReID={reid_thresh}, MaxDisap={max_disap}")

# Display widgets
widgets.interact(update_parameters, 
                clip_thresh=clip_threshold_slider,
                reid_thresh=reid_threshold_slider, 
                max_disap=max_disappeared_slider)

print("\n💡 Adjust the sliders above to tune processing parameters")

## 📊 Export Results

In [ ]:
# Export processing results to JSON
if stats:
    results_data = {
        'video_path': VIDEO_PATH,
        'text_query': TEXT_QUERY,
        'output_path': OUTPUT_PATH,
        'processing_stats': stats,
        'configuration': {
            'clip_threshold': processor.clip_threshold,
            'reid_threshold': processor.reid_threshold,
            'max_disappeared': processor.max_disappeared,
            'feature_memory_size': processor.feature_memory_size
        },
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    
    # Save to JSON
    results_file = 'processing_results.json'
    with open(results_file, 'w') as f:
        json.dump(results_data, f, indent=2)
    
    print(f"📊 Results exported to: {results_file}")
    
    # Display summary
    print("\n📋 FINAL SUMMARY")
    print("="*30)
    print(f"🎬 Video: {Path(VIDEO_PATH).name}")
    print(f"🎯 Query: '{TEXT_QUERY}'")
    print(f"📁 Output: {OUTPUT_PATH}")
    print(f"⏱️  Time: {stats['processing_time']:.2f}s")
    print(f"🎯 Matches: {stats['clip_matches']}")
    print(f"🆔 Tracks: {stats['unique_tracks']}")
    print(f"🚀 FPS: {stats['fps']:.2f}")
    
else:
    print("❌ No results to export")

## 🎉 Conclusion

**Congratulations!** You've successfully processed a video with CLIP-based person tracking.

### What you accomplished:
- ✅ Loaded and processed a video clip
- ✅ Used natural language to describe target persons
- ✅ Tracked multiple entities with consistent IDs
- ✅ Generated an output video with tracking visualization
- ✅ Analyzed processing statistics and performance

### Next steps:
- 🔄 Try different text queries
- 🎛️ Experiment with parameter tuning
- 📹 Process different video clips
- 🎯 Combine multiple queries for comprehensive analysis

**Happy tracking! 🚀**